# A. Introduction

Ce notebook présente une solution pour la capture de la provenance des données dans les pipelines de préparation de données. La capture de la provenance est essentielle pour comprendre comment les enregistrements dans un jeu de données résultant d’une transformation ou d’une opération complexe sont reliés à leurs origines dans les jeux de données d’entrée.

### Objectifs

L’objectif principal est de développer une classe Python, appelée `TensorProv`, capable de :

1. Capturer la provenance des données pour les types d’opérations courantes.
2. Représenter cette provenance à l’aide de tenseurs binaires creux (_sparse tensors_) pour assurer une gestion mémoire et un traitement efficace.
3. Tester et évaluer les performances des approches implémentées, en mesurant notamment les temps d’exécution et l’utilisation des ressources.

### Définition des Méthodes de Capture de Provenance

Nous présentons dans ce notebook la capture de provenance de type "lineage" (ou provenance des lignées). Elle permet de suivre l'origine des données et de reconstruire leur parcours dans un processus de transformation ou d'analyse. Ce type de provenance documente les relations entre les entrées, les transformations appliquées, et les sorties d'un pipeline de traitement.


Nous avons eu deux approches d'implémentations :

1. **Lineage Simple** : Identification de la Ligne Principale. Cette approche se concentre sur l'identification unique de la ligne principale dans le DataFrame source qui a contribué à chaque ligne du DataFrame de sortie.
Cette méthode permet de :
- Simplifier la traçabilité des données.
- Identifier rapidement l'origine principale de chaque élément en sortie.
Exemple : Pour une transformation simple (par exemple, un filtrage ou une sélection de colonnes), chaque ligne en sortie est directement mappée à une ligne unique dans le DataFrame source.

2. **Lineage Étendu** : Identification de Toutes les Lignes Contributrices. Nous avons tenté par la suite une approche complémentaire pour capturer la totalité des lignes d'origine ayant contribué à chaque ligne en sortie. Cette méthode est particulièrement utile dans des opérations complexes comme l'agrégation, où plusieurs lignes d'entrée peuvent influencer une seule ligne en sortie.
Exemple : Dans une opération d'agrégation, comme une somme ou une moyenne par groupe, la provenance inclut l'ensemble des lignes du DataFrame source ayant été utilisées pour produire le résultat final.


Deux méthodes sont explorées pour capturer la provenance des données :

1. **Méthode par hashing**  
   Cette méthode consiste à attribuer une clé unique à chaque enregistrement des jeux de données d’entrée en utilisant une fonction de hachage. Ces clés servent ensuite à identifier les correspondances entre les enregistrements des jeux de données de sortie et d’entrée. Bien que cette approche soit efficace pour tracer les liens entre les données, elle peut être coûteuse en termes de calcul pour les grands volumes de données.

2. **Méthode par ajout d’ID**  
   Une colonne supplémentaire contenant des identifiants uniques est ajoutée aux jeux de données d’entrée. Ces identifiants permettent de tracer directement chaque enregistrement des jeux d’entrée vers le jeu de données de sortie. Cette méthode est plus légère en termes de calcul mais requiert des modifications du schéma des données.


### Définition des Opérations Clés

#### **1. Data Transformation**
Les transformations de données consistent à modifier les valeurs spécifiques des attributs d’un jeu de données sans en changer la structure ni le nombre d’enregistrements. Ces transformations peuvent inclure des opérations telles que :
- La **normalisation**, qui ajuste les valeurs pour les ramener dans une échelle spécifique.
- La **discrétisation**, qui divise les valeurs continues en intervalles discrets.
- La **binarisation**, qui convertit les valeurs en données binaires (par exemple, 1 ou 0).

#### **2. Vertical Data Reduction**
Les opérations de réduction verticale consistent à supprimer des colonnes (ou attributs) d’un jeu de données. Cela peut inclure :
- **Feature Selection (sélection des variables)** : Identifier et conserver uniquement les colonnes pertinentes pour notre analyse.
- **Drop Columns (suppression des colonnes)** : Retirer des colonnes inutiles ou redondantes.

Le résultat est un jeu de données avec moins d’attributs mais contenant toujours le même nombre d’enregistrements.

#### **3. Horizontal Data Reduction**
La réduction horizontale vise à filtrer les lignes d’un jeu de données. Cela inclurs :
- **Filtrage** : Supprimer les lignes qui ne remplissent pas certaines conditions (par exemple, supprimer les lignes avec des valeurs manquantes ou aberrantes).
- **Undersampling (sous-échantillonnage)** : Réduire le nombre de lignes pour équilibrer les classes dans un problème de classification.
- **Row Deletion (suppression des lignes)** : Retirer explicitement certaines lignes.

Le résultat est un sous-ensemble des enregistrements d’origine.

#### **4. Vertical Data Augmentation**
L’augmentation verticale modifie le schéma d’un jeu de données sans en changer le nombre de lignes. Cela inclut :
- **Space Transformation** : Projeter les données dans un nouvel espace de caractéristiques (par exemple, utiliser une transformation PCA).
- **String Indexer** : Convertir les valeurs catégoriques en indices numériques.
- **One-Hot Encoding** : Créer des colonnes binaires pour représenter des catégories.

#### **5. Horizontal Data Augmentation**
L’augmentation horizontale génère de nouvelles lignes dans un jeu de données. Les opérations typiques incluent :
- **Oversampling (suréchantillonnage)** : Ajouter des duplications ou interpolations pour équilibrer les classes dans les données.
- **Instance Generation** : Générer artificiellement des enregistrements supplémentaires à partir des données existantes.

#### **6. Join (Fusion des Données)**
L’opération de jointure combine deux jeux de données en fonction d’une ou plusieurs colonnes communes. Cela inclut des types de jointures tels que :
- **Inner Join** : Conserver uniquement les enregistrements correspondants dans les deux jeux.
- **Left/Right Join** : Conserver tous les enregistrements d’un jeu de données et les correspondances dans l’autre.
- **Full Outer Join** : Conserver tous les enregistrements des deux jeux, en remplissant les valeurs manquantes avec `NULL`.

#### **7. Append (Concaténation)**
L’opération d’append ajoute les enregistrements d’un jeu de données à un autre. Contrairement à la jointure, les colonnes des deux jeux n’ont pas besoin d’être identiques :
- Les colonnes absentes sont complétées par des valeurs `NULL`.
- Les enregistrements du second jeu sont ajoutés à la fin du premier.

Cette opération est souvent utilisée pour assembler des ensembles de données qui partagent une structure similaire ou pour consolider des données provenant de différentes sources.


# B. Implémentation de la classe TensorProv

### 1. Implémentation de classe TensorProv

#### 1.1. Objectif Principal
La classe TensorProv est conçu pour capturer et documenter la provenance des transformations de données. Elle remplie principalement en trois objectifs :
- Identifier les relations entre les données sources et les données résultantes.
- Générer des matrices de provenance (sous forme de matrices creuses COO) pour retracer les liens entre les lignes des DataFrames d'entrée et de sortie.
- Gérer plusieurs types de provenance, selon la nature de la transformation.

#### 1.2. Types de Provenance Gérés
Nous avons identifié dans notre projet quatre types principaux de provenance. Les types 1, 2 et 3 sont des implémentations de type lineage simple. C'est à dire qu'une ligne du dataframe résultat est toujours liée à une ligne d'un dataframe source. Et cela même s'il y'a plusieurs dataframes sources. Le type 4 est une implémentation que nous présentons pour tenter de capturer un lineage étendu (voir définition plus haut), nous tenterons par cette implémentation de capturer la provenance d'un ligne résultat dans plusieurs lignes sources.

- **Type 1 (Lineage Simple)** : Une ligne en sortie provient d'une seule ligne en entrée (ex : filtrage, suppression de colonnes). Une seule matrice de provenance est construire.
- **Type 2 (Lineage Simple)** : Une ligne en sortie provient d'une ligne dans chacun des deux DataFrames d'entrée (ex : jointure). Une liste de matrice de provenance est construite. Une matrice par ligne résultat. Dans chaque matrice on capture la provenance entre les deux dataframe source. Il s'agit la d'une implémentation qui tente de reprendre le slide 12 du document du projet.
- **Type 3 (Lineage Simple)** : Une ligne en sortie est dérivée de plusieurs lignes dans un seul DataFrame d'entrée (ex : concaténation, agrégation). Le type 3 est un proposition d'implémentation différente pour la capture de provenance avec 2 dataframe d'origine. Une liste de matrice de provenance est construite. Chaque matrice capture la relation entre un dataframe d'origine et le dataframe résultat. Un peu comme pourrait le faire le type 1. Contrairement au Type2 qui ne peut capturer une provenance que pour 2 dataframes sources, l'implémentation de type 3 est capable de capturer la provenance de n dataframe source.
- **Type 4 (Lineage Etendu)** : Une ligne en sortie est générée à partir d'un regroupement par des lignes multiples (ex : group_by). Par ce 4eme type, nous tentons d'aller un peu plus loin que la demande initiale du projet en capturant une provenance multi-lignes dans le dataframe d'origine.

<div class="alert alert-block alert-danger">
<b>Commentaire:</b> Contrairement au type 2 qui ne peut capturer la provenance que pour une opération de jointure (merge), la type 3 est capable de le faire pour l'opération de jointure et l'opération de concaténation (append).
</div>

#### 1.3. Valeurs de retour de la méthode __call__

La méthode __call__ de la classe TensorProv retourne un tuple contenant trois éléments :

1. **result_df (pd.DataFrame)** :
Il s'agit du DataFrame transformé, obtenu après l'exécution de la fonction encapsulée.
Ce DataFrame représente le résultat de l'application de la fonction sur le(s) DataFrame(s) d'entrée, avec les modifications, filtrages, transformations ou agrégations effectuées.

2. **execution_time (float)** :
Cette valeur représente le temps total d'exécution du suivi de provenance, exprimé en secondes.
Il est calculé comme (end_time - start_time), où start_time est enregistré avant l'exécution de la fonction et end_time après la construction de la matrice de provenance.
Cette mesure permet d'évaluer l'impact de la capture de provenance sur les performances.

3. **provenance_matrix (scipy.sparse.coo_matrix ou liste de matrices)** :
Il s'agit d'une matrice creuse au format COO (Coordinate Format) qui capture les relations de provenance entre les lignes du DataFrame d'entrée et celles du DataFrame de sortie. La dimension de la matrice dépend du type de provenance utilisé :
Pour les types 1, 3 et 4, il s'agit d'une matrice unique de taille (m, n), où :
m = nombre de lignes du DataFrame de sortie.
n = nombre de lignes du DataFrame d'entrée.
Pour le type 2, une liste de matrices est retournée, où chaque matrice représente une ligne du DataFrame de sortie et capture sa relation avec les lignes des deux DataFrames d'entrée.
La matrice contient des valeurs binaires (1 ou 0), où 1 signifie qu'une ligne du DataFrame de sortie provient d'une ligne du DataFrame d'entrée.

<div class="alert alert-block alert-info">
Ce format de retour permet à <b>TensorProv</b> de tracer et documenter l'origine des transformations de données, tout en offrant une mesure des performances via le temps d'exécution.
</div>


In [ ]:
import pandas as pd
import numpy as np
from scipy.sparse import (coo_matrix)
import hashlib
import time

class TensorProv:
    """
    TensorProv is designed to capture and document the provenance of data transformations.
    Its main objectives are:
    1. Identify the relationships between source data and resulting data.
    2. Generate provenance matrices (in sparse COO format) to trace the links between rows in
       input DataFrames and the output DataFrame.
    3. Handle multiple types of provenance depending on the nature of the transformation.

    ### Types of Provenance Supported
    The project supports four main types of provenance:
    1. **Type 1 (Simple Lineage)**: Each row in the output corresponds to a single row in
       the input (e.g., filtering, column removal). One provenance matrix is constructed.
    2. **Type 2 (Simple Lineage)**: Each row in the output corresponds to one row from
       two input DataFrames (e.g., join operations). A list of provenance matrices is created,
       one matrix per output row, capturing the relationship between the two source DataFrames.
    3. **Type 3 (Simple Lineage)**: Each row in the output is derived from multiple rows
       in a single input DataFrame (e.g., concatenation, aggregation). A list of matrices
       is created, with each matrix capturing the relationship between one source DataFrame
       and the resulting DataFrame.
    4. **Type 4 (Extended Lineage)**: An output row is generated by grouping multiple rows
       from the input DataFrame (e.g., group by operations). This goes beyond the initial
       project requirements, as it captures multi-lineage provenance in the source DataFrame.
    """
    def __init__(self, function, method='hash'):
        """
        Initialize the TensorProv object.
        Creates a mapping between the names of the functions and the types of provenance.

        Parameters:
        function (callable): The function to be wrapped for provenance tracking.
        method (str): The method to use for provenance tracking ('hash' or 'ids').
        """
        self.function = function
        self.function_name = self.function.__name__
        self.method = method

        self.prov_type1_list = ['query_func','reduce_func', 'oversampling_func', 'one_hot_encoder_func', 'drop_columns_func',
                                'drop_rows_func', 'fill_na_func']
        self.prov_type2_list = ['merge_func']
        self.prov_type3_list = ['append_func']
        self.prov_type4_list = ['group_by_func']

    def __call__(self, *args, **kwargs):
        """
        Call the wrapped function and capture provenance based on the function type.

        Parameters:
        *args: Positional arguments for the wrapped function.
        **kwargs: Keyword arguments for the wrapped function.

        Returns:
        The result of the wrapped function with provenance information.
        """
        args, kwargs = TensorProv.check_call_args(self, *args, **kwargs)

        if self.function_name in self.prov_type1_list:
            return self.prov_type1(*args, **kwargs)
        elif self.function_name in self.prov_type2_list:
            return self.prov_type2(*args, **kwargs)
        elif self.function_name in self.prov_type3_list:
            return self.prov_type3(*args, **kwargs)
        elif self.function_name in self.prov_type4_list:
            return self.prov_type4(*args, **kwargs)
        else:
            raise ValueError(f"Cannot capture provenance for more than 2 dataframes.")

    @staticmethod
    def hash_row_content(row: pd.Series) -> str:
        """
        Generate a hash for the content of a row.

        Parameters:
        row (pd.Series): The row to hash.

        Returns:
        str: The hash of the row content.
        """

        row_str = row.to_json(date_format="iso", orient="columns")
        return hashlib.md5(row_str.encode("utf-8")).hexdigest()

    def check_call_args(self, *args, **kwargs):
        """
        Check and modify the call arguments based on the method.

        Parameters:
        *args: Positional arguments for the wrapped function.
        **kwargs: Keyword arguments for the wrapped function.

        Returns:
        tuple: Modified positional and keyword arguments.
        """

        if "reduced_columns" in kwargs:
            reduced_columns = kwargs["reduced_columns"]
            if isinstance(reduced_columns, list):
                if self.method == "hash" and "_hash_" not in reduced_columns:
                    reduced_columns.append("_hash_")
                if self.method == "ids" and "_id_" not in reduced_columns:
                    reduced_columns.append("_id_")
        if "group_by_column" in kwargs:
            if self.method == "hash":
                indices_args = {'prov_column':'_hash_'}
                kwargs = {k: v for d in (kwargs, indices_args) for k, v in d.items()}
            if self.method == "ids":
                indices_args = {'prov_column': '_id_'}
                kwargs = {k: v for d in (kwargs, indices_args) for k, v in d.items()}

        return args, kwargs


    def prov_type1 (self, *args, **kwargs):
        """
        Handles Type 1 provenance: One-to-One Relationships.

        Parameters:
        *args: Positional arguments for the wrapped function.
        **kwargs: Keyword arguments for the wrapped function.

        Returns:
        The result of the wrapped function with provenance information.
        """
        if self.method == 'ids':
            return self.ids_prov_type1(*args, **kwargs)
        else:
            return self.hash_prov_type1(*args, **kwargs)

    def hash_prov_type1(self, *args, **kwargs):
        """
        Captures Type 1 provenance using hash-based tracking.

        ### Description:
        - This method tracks provenance when each row in the output DataFrame corresponds to
          exactly one row in the input DataFrame.
        - Instead of using unique row indices, it generates **hashes** to uniquely identify
          each row, ensuring robustness even if row positions change.

        ### Steps:
        1. Extract the input DataFrame (`df`) from `kwargs`.
        2. Generate a unique hash (`_hash_`) for each row in the input DataFrame.
        3. Execute the wrapped function to obtain the transformed DataFrame (`result_df`).
        4. Create a dictionary mapping `_hash_` values to row positions in the input DataFrame.
        5. Iterate over `result_df` to establish row mappings based on hashes.
        6. Construct a sparse COO matrix linking input rows to output rows.

        ### Returns:
        - result_df: The transformed DataFrame.
        - execution_time: The time taken to compute provenance.
        - provenance_matrix: A sparse COO matrix capturing row relationships.
        """

        start_time = time.time()  # Start time tracking

        # Step 1: Retrieve input DataFrame
        df = kwargs.get('df')
        if df is None:
            raise ValueError("Input DataFrame ('df') is required.")

        # Step 2: Generate a unique hash for each row in the input DataFrame
        df["_hash_"] = df.apply(TensorProv.hash_row_content, axis=1)

        # Step 3: Execute the wrapped function to get the result DataFrame
        result_df = self.function(**kwargs).copy()

        # Step 4: Define input and output DataFrame sizes
        n = len(df)  # Number of rows in the original DataFrame
        m = len(result_df)  # Number of rows in the transformed DataFrame

        # Step 5: Create a dictionary mapping `_hash_` values to their original row positions
        hash_origin_with_position = {h: i for i, h in enumerate(df["_hash_"].values)}

        # Step 6: Initialize lists to store row mappings
        row_positions = []  # Stores output row indices
        col_positions = []  # Stores corresponding input row indices

        # Step 7: Iterate over result_df to establish provenance mapping
        for i, hash_result in enumerate(result_df["_hash_"]):
            if hash_result not in hash_origin_with_position:
                # If a hash in the result_df does not exist in the input DataFrame, raise an error
                raise ValueError(f"Hash {hash_result} in result_df not found in original_df. "
                                 f"Possible data mismatch or missing row?")
            else:
                j = hash_origin_with_position[hash_result]  # Retrieve input row index
                row_positions.append(i)  # Output row index
                col_positions.append(j)  # Corresponding input row index

        # Step 8: Construct the sparse COO matrix
        data = np.ones(len(col_positions), dtype=np.int8)  # All connections are marked as '1'

        provenance_matrix = coo_matrix(
            (data, (row_positions, col_positions)),  # Non-zero positions
            shape=(m, n)  # Dimensions of the provenance matrix
        )

        end_time = time.time()  # End time tracking

        # Step 9: Return the transformed DataFrame, execution time, and provenance matrix
        return result_df, (end_time - start_time), provenance_matrix

    def ids_prov_type1(self, *args, **kwargs):
        """
        Captures Type 1 provenance using ID-based tracking.

        ### Description:
        - This method tracks provenance when each row in the output DataFrame corresponds to
          exactly one row in the input DataFrame.
        - Instead of using hash values, it assigns unique IDs to rows and uses them to construct
          a sparse COO provenance matrix.

        ### Steps:
        1. Extract the input DataFrame (`df`) from `kwargs`.
        2. Assign a unique identifier (`_id_`) to each row based on its index.
        3. Execute the wrapped function to obtain the transformed DataFrame (`result_df`).
        4. Build a mapping between `_id_` values and row positions in the input DataFrame.
        5. Iterate over `result_df` to establish row mappings.
        6. Construct a sparse COO matrix linking input rows to output rows.

        ### Returns:
        - result_df: The transformed DataFrame.
        - execution_time: The time taken to compute provenance.
        - provenance_matrix: A sparse COO matrix capturing row relationships.
        """

        start_time = time.time()  # Start time tracking

        # Step 1: Retrieve input DataFrame
        df = kwargs.get('df')
        if df is None:
            raise ValueError("Input DataFrame ('df') is required.")

        # Step 2: Assign unique IDs to each row based on index
        df["_id_"] = df.index

        # Step 3: Execute the wrapped function
        result_df = self.function(**kwargs).copy()

        # Step 4: Define input and output DataFrame sizes
        n = len(df)  # Number of rows in input DataFrame
        m = len(result_df)  # Number of rows in output DataFrame

        # Step 5: Create a dictionary mapping `_id_` values to their original row positions
        id_origin_ids_with_position = {h: i for i, h in enumerate(df["_id_"].values)}

        # Step 6: Retrieve `_id_` values from the output DataFrame
        result_ids = result_df['_id_'].values

        # Step 7: Initialize lists to store row mappings
        row_positions = []
        col_positions = []

        # Step 8: Iterate over result_df to build the provenance mapping
        for i, result_id in enumerate(result_ids):
            # Find the corresponding position in the original DataFrame
            j = id_origin_ids_with_position[result_id]
            row_positions.append(i)  # Output row index
            col_positions.append(j)  # Corresponding input row index

        # Step 9: Construct the sparse COO matrix
        data = np.ones(len(row_positions), dtype=np.int8)  # Values in the matrix

        provenance_matrix = coo_matrix(
            (data, (row_positions, col_positions)),  # Non-zero positions
            shape=(m, n)  # Dimensions of the provenance matrix
        )

        end_time = time.time()  # End time tracking

        # Step 10: Return the transformed DataFrame, execution time, and provenance matrix
        return result_df, (end_time - start_time), provenance_matrix

    def prov_type2(self, *args, **kwargs):
        """
        Handles Type 2 provenance: One output row corresponds to one row in each of two input DataFrames.

        Parameters:
        *args: Positional arguments for the wrapped function.
        **kwargs: Keyword arguments for the wrapped function.

        Returns:
        The result of the wrapped function with provenance information.
        """

        # Step 1: Determine whether to use ID-based or hash-based tracking
        if self.method == 'ids':
            return self.ids_prov_type2(*args, **kwargs)  # Call ID-based provenance function
        else:
            return self.hash_prov_type2(*args, **kwargs)  # Call hash-based provenance function

    def hash_prov_type2(self, *args, **kwargs):
        """
        Captures Type 2 provenance using hash-based tracking.

        ### Description:
        - This method tracks provenance when each row in the output DataFrame corresponds to
          one row in each of two input DataFrames.
        - Instead of using unique row indices, it generates **hashes** to uniquely identify
          each row, ensuring robustness even if row positions change.
        - It constructs **a list of sparse COO matrices**, where each matrix corresponds to
          a row in the output DataFrame and links it to rows in both input DataFrames.

        ### Steps:
        1. Extract the input DataFrames (`df1` and `df2`) from `kwargs`.
        2. Generate a unique hash (`_hash_1` and `_hash_2`) for each row in both input DataFrames.
        3. Execute the wrapped function to obtain the transformed DataFrame (`result_df`).
        4. Create dictionaries mapping `_hash_1` and `_hash_2` values to row positions.
        5. Iterate over `result_df` to establish row mappings based on hashes.
        6. Construct **a separate sparse COO matrix for each output row** linking it to input rows.
        7. Append each provenance matrix to a list and return the results.

        ### Returns:
        - result_df: The transformed DataFrame.
        - execution_time: The time taken to compute provenance.
        - t: A list of sparse COO matrices capturing row relationships.
        """

        # Step 1: Retrieve the two input DataFrames from kwargs
        df1 = kwargs.get('df1')
        if df1 is None:
            raise ValueError("Call argument 'df1' is required.")

        df2 = kwargs.get('df2')
        if df2 is None:
            raise ValueError("Call argument 'df2' is required.")

        # Step 2: Generate unique hash identifiers for each row in both DataFrames
        df1["_hash_1"] = df1.apply(TensorProv.hash_row_content, axis=1)
        df2["_hash_2"] = df2.apply(TensorProv.hash_row_content, axis=1)

        # Step 3: Execute the wrapped function to get the result DataFrame
        result_df = self.function(**kwargs).copy()

        # Step 4: Initialize list to store provenance matrices and track execution time
        t = []
        start_time = time.time()

        # Step 5: Create dictionaries mapping hashes to original row positions
        hash_1_origin_with_position = {h: i for i, h in enumerate(df1["_hash_1"].values)}
        hash_2_origin_with_position = {h: i for i, h in enumerate(df2["_hash_2"].values)}

        # Step 6: Iterate over result_df to establish provenance mapping
        for index, row in result_df.iterrows():
            # Retrieve the hash values from the result DataFrame
            row_hash_1 = row["_hash_1"]
            row_hash_2 = row["_hash_2"]

            # Find the corresponding row positions in the original DataFrames
            index_1 = hash_1_origin_with_position.get(row_hash_1)
            index_2 = hash_2_origin_with_position.get(row_hash_2)

            # Step 7: Create a sparse COO matrix linking this output row to input rows
            row_positions = [index_1]  # Output row index
            col_positions = [index_2]  # Corresponding input row index
            data = [1]  # Provenance relationship is binary (1 means linked)

            provenance_matrix_df = coo_matrix(
                (data, (row_positions, col_positions)),  # Non-zero positions
                shape=(len(df1), len(df2))  # Matrix dimensions: Input DataFrame sizes
            )

            # Step 8: Append the matrix to the list
            t.append(provenance_matrix_df)

        end_time = time.time()  # End time tracking

        # Step 9: Return the transformed DataFrame, execution time, and list of provenance matrices
        return result_df, (end_time - start_time), t

    def ids_prov_type2(self, *args, **kwargs):

        df1 = kwargs.get('df1')
        if df1 is None:
            raise ValueError(f"Call argument 'df1' is required.")
        df2 = kwargs.get('df2')
        if df2 is None:
            raise ValueError(f"Call argument 'df2' is required.")

        df1["_id_1"] = df1.index
        df2["_id_2"] = df2.index
        result_df = self.function(**kwargs).copy()

        id_1_origin_ids_with_position = {
            h: i for i, h in enumerate(df1["_id_1"].values)
        }
        id_2_origin_ids_with_position = {
            h: i for i, h in enumerate(df2["_id_2"].values)
        }

        result_df = self.function(**kwargs).copy()
        t = []
        start_time = time.time()

        for index, row in result_df.iterrows():
            row_id_1 = row["_id_1"]
            row_id_2 = row["_id_2"]

            index_1 = id_1_origin_ids_with_position.get(row_id_1)
            index_2 = id_2_origin_ids_with_position.get(row_id_2)

            row_positions = [index_1]
            col_positions = [index_2]
            data = [1]

            provenance_matrix_df = coo_matrix(
                (data, (row_positions, col_positions)),
                shape=(len(df1), len(df2))
            )
            t.append(provenance_matrix_df)

        end_time = time.time()

        return result_df, (end_time - start_time), t

    @staticmethod
    def get_dataframes_from_kwargs(**kwargs):
        """
        Extracts all DataFrame-type arguments from the provided keyword arguments.

        ### Description:
        - This utility method scans through the keyword arguments (`kwargs`) and filters out
          all values that are instances of `pd.DataFrame`.
        - It is used in provenance tracking methods to extract DataFrames from function arguments
          without explicitly knowing their names.

        ### Steps:
        1. Iterate over the values of `kwargs`.
        2. Check if each value is an instance of `pd.DataFrame`.
        3. Collect all DataFrames into a list.
        4. Return the list of extracted DataFrames.

        ### Returns:
        - A **list of DataFrames** found in `kwargs`.
          - If no DataFrames are present, an empty list is returned.

        ### Example:
        ```python
        kwargs = {"df1": pd.DataFrame(), "df2": pd.DataFrame(), "param": 5}
        dfs = TensorProv.get_dataframes_from_kwargs(**kwargs)
        print(len(dfs))  # Output: 2 (Only DataFrames are extracted)
        ```
        """

        # Step 1: Iterate over kwargs values and extract only the ones that are DataFrames
        dataframes = [value for value in kwargs.values() if isinstance(value, pd.DataFrame)]

        # Step 2: Return the extracted DataFrames
        return dataframes

    def prov_type3(self, *args, **kwargs):
        """
        Handles Type 3 provenance: One output row is derived from one row in multiple input DataFrame.

        Parameters:
        *args: Positional arguments for the wrapped function.
        **kwargs: Keyword arguments for the wrapped function.

        Returns:
        The result of the wrapped function with provenance information.
        """

        # Step 1: Determine whether to use ID-based or hash-based tracking
        if self.method == 'ids':
            return self.ids_prov_type3(*args, **kwargs)  # Call ID-based provenance function
        else:
            return self.hash_prov_type3(*args, **kwargs)  # Call hash-based provenance function

    def hash_prov_type3(self, *args, **kwargs):
        """
        Captures Type 3 provenance using hash-based tracking.

        ### Description:
        - Type 3 provenance is used when One output row is derived from one row in multiple input DataFrame.
        - This commonly occurs in operations like **concatenation, aggregation, or transformations**
          where multiple input rows contribute to a single output row.
        - Instead of using row indices, this method generates **hashes** to uniquely identify
          rows and track their provenance.
        - A **list of sparse COO matrices** is created, where each matrix corresponds to
          an input DataFrame and captures its relationship with the output DataFrame.

        ### Steps:
        1. **Extract DataFrames** from function arguments (`kwargs`).
        2. **Generate a unique hash (`_hash_1`, `_hash_2`, etc.)** for each row in the input DataFrames.
        3. **Execute the wrapped function** to obtain the transformed DataFrame (`result_df`).
        4. **Iterate over the input DataFrames** and build provenance mappings:
           - Create a **dictionary mapping hash values to row positions**.
           - Iterate over `result_df` and determine the corresponding row positions in the input DataFrames.
           - Construct **a sparse COO matrix** capturing these relationships.
        5. **Append each provenance matrix to a list** and return the results.

        ### Returns:
        - `result_df`: The transformed DataFrame.
        - `execution_time`: The time taken to compute provenance.
        - `t`: A **list of sparse COO matrices**, where each matrix tracks the relationship
          between an input DataFrame and the output DataFrame.
        """

        # Step 1: Extract DataFrames from function arguments
        dataframes = TensorProv.get_dataframes_from_kwargs(**kwargs)

        # Step 2: Generate unique hash identifiers for each row in the input DataFrames
        for index, dataframe in enumerate(dataframes, start=1):
            dataframe[f"_hash_{index}"] = dataframe.apply(TensorProv.hash_row_content, axis=1)

        # Step 3: Execute the wrapped function to get the result DataFrame
        result_df = self.function(**kwargs).copy()

        # Step 4: Initialize list to store provenance matrices and start execution timer
        t = []
        start_time = time.time()

        # Step 5: Iterate over input DataFrames and establish provenance mapping
        for index, dataframe in enumerate(dataframes, start=1):
            dataframe_len: int = len(dataframe)  # Total rows in the input DataFrame
            result_df_len: int = len(result_df)  # Total rows in the output DataFrame

            # Step 5.1: Create a dictionary mapping hash values to their original row positions
            hash_to_position = {h: i for i, h in enumerate(dataframe[f"_hash_{index}"].values)}

            # Step 5.2: Initialize arrays for provenance matrix
            row_positions = np.arange(result_df_len, dtype=np.int32)  # Output row indices
            col_positions = np.zeros(result_df_len, dtype=np.int32)  # Default: No mapping found
            data = np.zeros(result_df_len, dtype=np.int32)  # Default: No relationship

            # Step 5.3: Iterate over result_df to establish provenance relationships
            for i in range(result_df_len):
                row_hash = result_df.iloc[i][f"_hash_{index}"]
                if not (pd.isna(row_hash) or row_hash not in hash_to_position):
                    orig_pos = hash_to_position[row_hash]  # Retrieve input row index
                    col_positions[i] = orig_pos  # Map output row to input row
                    data[i] = 1  # Relationship exists

            # Step 5.4: Construct the sparse COO matrix for provenance tracking
            provenance_matrix_df = coo_matrix(
                (data, (row_positions, col_positions)),  # Non-zero positions
                shape=(result_df_len, dataframe_len)  # Matrix dimensions: (output rows, input rows)
            )

            # Step 5.5: Append the matrix to the list
            t.append(provenance_matrix_df)

        end_time = time.time()  # End time tracking

        # Step 6: Return the transformed DataFrame, execution time, and list of provenance matrices
        return result_df, (end_time - start_time), t

    def ids_prov_type3(self, *args, **kwargs):
        """
        Captures Type 3 provenance using ID-based tracking.

        ### Description:
        - Type 3 provenance is used when One output row is derived from one row in multiple input DataFrame.
        - This commonly occurs in operations like **concatenation, aggregation, or transformations**
          where multiple input rows contribute to a single output row.
        - Instead of using **hashes**, this method assigns unique **IDs (`_id_1`, `_id_2`, etc.)**
          to track rows and establish provenance relationships.
        - A **list of sparse COO matrices** is created, where each matrix corresponds to
          an input DataFrame and captures its relationship with the output DataFrame.

        ### Steps:
        1. **Extract DataFrames** from function arguments (`kwargs`).
        2. **Assign unique IDs (`_id_1`, `_id_2`, etc.)** to each row in the input DataFrames.
        3. **Execute the wrapped function** to obtain the transformed DataFrame (`result_df`).
        4. **Iterate over the input DataFrames** and build provenance mappings:
           - Extract `_id_` values from `result_df`.
           - Convert `NaN` values to zeros and ensure valid integer indices.
           - Construct **a sparse COO matrix** capturing these relationships.
        5. **Append each provenance matrix to a list** and return the results.

        ### Returns:
        - `result_df`: The transformed DataFrame.
        - `execution_time`: The time taken to compute provenance.
        - `t`: A **list of sparse COO matrices**, where each matrix tracks the relationship
          between an input DataFrame and the output DataFrame.
        """

        # Step 1: Extract DataFrames from function arguments
        dataframes = TensorProv.get_dataframes_from_kwargs(**kwargs)

        # Step 2: Assign unique ID numbers to each row in the input DataFrames
        for index, dataframe in enumerate(dataframes, start=1):
            dataframe[f"_id_{index}"] = np.arange(len(dataframe))

        # Step 3: Execute the wrapped function to get the result DataFrame
        result_df = self.function(**kwargs).copy()

        # Step 4: Initialize list to store provenance matrices and start execution timer
        t = []
        start_time = time.time()

        # Step 5: Iterate over input DataFrames and establish provenance mapping
        for index, dataframe in enumerate(dataframes, start=1):
            dataframe_len: int = len(dataframe)  # Total rows in the input DataFrame
            result_df_len: int = len(result_df)  # Total rows in the output DataFrame

            # Step 5.1: Create row position mappings
            row_positions = np.arange(result_df_len)  # Output row indices
            orig_positions = result_df[f"_id_{index}"].to_numpy()  # Extract `_id_` values from result_df

            # Step 5.2: Convert `NaN` values to zeros and ensure valid integer indices
            col_positions = np.nan_to_num(orig_positions, nan=0).astype(int)

            # Step 5.3: Create a binary mapping (1 for linked rows, 0 for unlinked)
            data = np.where(np.isnan(orig_positions), 0, 1).astype(np.int8)

            # Step 5.4: Construct the sparse COO matrix for provenance tracking
            provenance_matrix = coo_matrix(
                (data, (row_positions, col_positions)),  # Non-zero positions
                shape=(result_df_len, dataframe_len)  # Matrix dimensions: (output rows, input rows)
            )

            # Step 5.5: Append the matrix to the list
            t.append(provenance_matrix)

        end_time = time.time()  # End time tracking

        # Step 6: Return the transformed DataFrame, execution time, and list of provenance matrices
        return result_df, (end_time - start_time), t

    def prov_type4 (self, *args, **kwargs):
        """
        Handles Type 4 provenance: One output row is derived from multiple rows in a single input DataFrame.

        Parameters:
        *args: Positional arguments for the wrapped function.
        **kwargs: Keyword arguments for the wrapped function.

        Returns:
        The result of the wrapped function with provenance information.
        """
        if self.method == 'ids':
            return self.ids_prov_type4(*args, **kwargs)
        else:
            return self.hash_prov_type4(*args, **kwargs)

    def hash_prov_type4(self, *args, **kwargs):
        """
        Captures Type 4 provenance using hash-based tracking.

        ### Description:
        - Type 4 provenance is used when **one output row is derived from multiple rows**
          in a **single input DataFrame**, typically in **grouping operations (`group_by`)**.
        - Instead of using **row indices**, this method generates **hashes** to uniquely identify
          rows and track their provenance.
        - A **single sparse COO matrix** is constructed to map output rows to their contributing
          input rows.

        ### Steps:
        1. **Start execution timer** to measure performance.
        2. **Retrieve the input DataFrame** from function arguments (`kwargs`).
        3. **Generate a unique hash (`_hash_`)** for each row in the input DataFrame.
        4. **Execute the wrapped function** to obtain the transformed DataFrame (`result_df`).
        5. **Create a dictionary mapping hash values to row positions** in the input DataFrame.
        6. **Initialize provenance matrix components**:
           - `row_positions`: Stores output row indices.
           - `col_positions`: Stores corresponding input row indices.
           - `data`: Stores binary values (1 if linked).
        7. **Iterate over `result_df`** and determine input row mappings.
        8. **Construct a sparse COO matrix** to capture provenance.
        9. **Return the provenance matrix and execution time**.

        ### Returns:
        - `provenance_matrix`: A sparse COO matrix mapping input rows to output rows.
        - `execution_time`: The time taken to compute provenance.
        """
        start_time = time.time()

        df = kwargs.get('df')
        if df is None:
            raise ValueError(f"Call argument 'df' is required.")

        df["_hash_"] = df.apply(TensorProv.hash_row_content, axis=1)
        result_df = self.function(**kwargs).copy()

        n: int = len(df)  # total rows in original df
        m: int = len(result_df)  # total rows in filtered df

        hash_to_position = {
            h: i for i, h in enumerate(df["_hash_"].values)
        }

        row_positions = np.empty(n, dtype=np.int32)
        col_positions = np.arange(n, dtype=np.int32)
        data = np.ones(n, dtype=np.int8)

        # For each row i in result_df, find the original row position via the hash
        for i in range(m):
            row_hash = result_df.iloc[i]["_hash_"]
            if isinstance(row_hash, list):
                for h in row_hash:
                    if h not in hash_to_position:
                        raise ValueError(f"Hash {h} in result_df not found in original_df. "
                                         f"Possible data mismatch or missing row?")
                    orig_pos = hash_to_position[h]
                    row_positions[orig_pos] = i
            else:
                raise ValueError(f"Unexpected type for hash value: {type(row_hash)}")

        provenance_matrix = coo_matrix(
            (data, (row_positions, col_positions)),
            shape=(m, n)
        )
        end_time = time.time()

        return result_df, (end_time - start_time), provenance_matrix

    def ids_prov_type4(self, *args, **kwargs):
        """
        Captures Type 4 provenance using ID-based tracking.

        ### Description:
        - Type 4 provenance is used when **one output row is derived from multiple rows**
          in a **single input DataFrame**, typically in **grouping operations (`group_by`)**.
        - Instead of using **hashes**, this method assigns **unique row IDs (`_id_`)**
          to track rows and establish provenance relationships.
        - A **single sparse COO matrix** is constructed to map output rows to their contributing
          input rows.

        ### Steps:
        1. **Start execution timer** to measure performance.
        2. **Retrieve the input DataFrame** from function arguments (`kwargs`).
        3. **Assign unique IDs (`_id_`)** to each row in the input DataFrame.
        4. **Execute the wrapped function** to obtain the transformed DataFrame (`result_df`).
        5. **Define input and output DataFrame sizes** (`n` for input, `m` for output).
        6. **Initialize provenance matrix components**:
           - `col_positions`: Stores input row indices.
           - `row_positions`: Stores output row indices.
           - `data`: Stores binary values (1 if linked).
        7. **Iterate over `result_df`** and determine input row mappings.
        8. **Construct a sparse COO matrix** to capture provenance.
        9. **Return the transformed DataFrame, execution time, and provenance matrix**.

        ### Returns:
        - `result_df`: The transformed DataFrame.
        - `execution_time`: The time taken to compute provenance.
        - `provenance_matrix`: A sparse COO matrix mapping input rows to output rows.
        """
        start_time = time.time()

        df = kwargs.get('df')
        if df is None:
            raise ValueError(f"Call argument 'df' is required.")

        df["_id_"] = df.index
        result_df = self.function(**kwargs).copy()

        n = len(df)
        m = len(result_df)

        col_positions = np.arange(n)
        row_positions = np.arange(n, dtype=np.int32)
        for index, ids in enumerate(result_df["_id_"]):
            row_positions[ids] = index

        data = np.ones(n, dtype=np.int8)

        # Build the sparse COO matrix
        provenance_matrix = coo_matrix(
            (data, (row_positions, col_positions)),
            shape=(m,n)
        )
        end_time = time.time()

        return result_df, (end_time - start_time), provenance_matrix

### 2. Implémentation de classe et methodes utilitaires

In [90]:
import random, string
import pandas as pd
import numpy as np
from typing import Literal
from imblearn.over_sampling import SMOTE


######### ---------------------------------------------------------------------------------------------------- #########
######### Méthodes de transformation que nous utiliserons dans la cadre des captures de provenance             #########
######### ---------------------------------------------------------------------------------------------------- #########


def query_function(df: pd.DataFrame, condition: str):
    return df.query(condition)


def reduce_function(df: pd.DataFrame, reduced_columns: list):
    return df[reduced_columns]


def hda_function(df: pd.DataFrame):
    x = df[['age']]
    y = df['city']
    x_resampled, y_resampled = SMOTE(k_neighbors=1).fit_resample(X, y)
    return pd.concat([
        pd.DataFrame(x_resampled, columns=x.columns),  # Caractéristiques augmentées
        pd.Series(y_resampled, name='city')  # Labels augmentés
    ],
        axis=1
    )
######### ---------------------------------------------------------------------------------------------------- #########
######### Méthodes utilitaires pour la génération de datafarme de tests                                        #########
######### ---------------------------------------------------------------------------------------------------- #########
def random_string(length=8):
    return ''.join(random.choices(string.ascii_letters + string.digits, k=length))

def generate_large_df(n_persons=1000):

    # Generate the person DataFrame
    persons_df = pd.DataFrame({
        "name": [random_string() for _ in range(n_persons)],
        "age": np.random.randint(18, 70, size=n_persons),
        "city": np.random.choice(["NY", "SF", "LA", "Berlin", "London"], size=n_persons)
    })

    hobbies_list = []
    for name in persons_df["name"]:
        n_hobbies = random.randint(0, 3)  # Randomly choose between 0 and 3 hobbies
        hobbies = np.random.choice(
            ["Reading", "Painting", "Cycling", "Cooking", "Gardening", "Hiking"],
            size=n_hobbies,
            replace=False
        )
        for hobby in hobbies:
            hobbies_list.append({"name": name, "hobbies": hobby})
    hobbies_df = pd.DataFrame(hobbies_list)

    return persons_df, hobbies_df

######### ---------------------------------------------------------------------------------------------------- #########
######### Mthode utilitaire permettant l'exceution des test et l'affichage d'n tableau récapitulatif           #########
######### ---------------------------------------------------------------------------------------------------- #########

def execute_test_and_log(df_counts: list, test_function):
    test_logs = []
    for count in df_counts:
        persons_df, hobbies_df = generate_large_df(n_persons=count)
        print(f"Running {test_function.__name__} tests with {count} rows")
        hash_tensor_filter = TensorProv(function= test_function, method='hash')
        df1, df2, result_df, execution_time, provenance_matrix_list  = hash_tensor_filter(df1=persons_df, df2=hobbies_df, on_merge="name", how_merge="inner")
        test_logs.append({
            "Test Name": f"Test {test_function.__name__} with {count} rows",
            "Capture Method": "Hash",
            "Original (Person) Rows": len(persons_df),
            "Original (Hobbies) Rows": len(hobbies_df), 
            "Transformation Function": test_function.__name__,
            "Result Rows": len(result_df),
            "Execution Time (s)": execution_time
        })
        ids_tensor_filter = TensorProv(function= test_function, method='ids')
        df1, df2, result_df, execution_time, provenance_matrix_list  = ids_tensor_filter(df1=persons_df, df2=hobbies_df, on_merge="name", how_merge="inner")
        test_logs.append({
            "Test Name": f"Test {test_function.__name__} with {count} rows",
            "Capture Method": "Ids",
            "Original (Person) Rows": len(persons_df),
            "Original (Hobbies) Rows": len(hobbies_df), 
            "Transformation Function": test_function.__name__,
            "Result Rows": len(result_df),
            "Execution Time (s)": execution_time
        })    

    df_logs = pd.DataFrame(test_logs)
    print("\nTest Summary:")
    print(df_logs.to_string(index=False))   

# C. Test de différentes transformation

### 1. Opération de fusion (merge)

In [ ]:

def merge_func(df1: pd.DataFrame, df2: pd.DataFrame, on_merge: str, how_merge: Literal["left", "right", "inner", "outer", "cross"] = "inner") -> pd.DataFrame:
    return df1.merge(df2, on=on_merge, how=how_merge, suffixes=('_x', '_y'))

df_counts = [10,100,1000,10000, 25000, 50000]
execute_test_and_log(df_counts, merge_func)


Running merge_func tests with 10 rows
Running merge_func tests with 100 rows
Running merge_func tests with 1000 rows
Running merge_func tests with 10000 rows
Running merge_func tests with 25000 rows
Running merge_func tests with 50000 rows
